In [1]:
import lxmls.readers.sentiment_reader as srs
from lxmls.deep_learning.utils import AmazonData
corpus = srs.SentimentCorpus("books")
data = AmazonData(corpus=corpus)

In [2]:
data.datasets['train']

{'input': array([[1., 1., 1., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 2., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 4., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]),
 'output': array([0, 1, 0, ..., 1, 0, 1])}

In [70]:
from lxmls.deep_learning.utils import Model, glorot_weight_init, index2onehot, logsumexp
import numpy as np

class NumpyLogLinear(Model):
    
    def __init__(self, **config):
        
        # Initialize parameters
        weight_shape = (config['input_size'], config['num_classes'])
        # after Xavier Glorot et al
        self.weight = glorot_weight_init(weight_shape, 'softmax')
        self.bias = np.zeros((1, config['num_classes']))
        self.learning_rate = config['learning_rate']
        
    def log_forward(self, input=None):  
        """Forward pass of the computation graph"""
        z =  input @ self.weight.T+ self.bias
        log_z_tilde = z - np.log(np.sum(np.exp(z), axis=1,keepdims=True))
        return log_z_tilde
        
    def predict(self, input=None):
        """Prediction: most probable class index"""
        return np.argmax(np.exp(self.log_forward(input)), axis=1)      

    def backpropagation(self, input, output):
        batch_size = len(input)
        log_z_tilde = self.log_forward(input)
        loss = - np.sum(log_z_tilde[np.arange(batch_size),output]) / batch_size
        z_tilde = np.exp(log_z_tilde)
        I = index2onehot(output, self.weight.shape[0])
        grad_wrt_z = I-z_tilde
        gradient_weight = - ((grad_wrt_z).T @ input)/batch_size
        gradient_bias = - np.sum(grad_wrt_z,0)/batch_size
        return loss, gradient_weight, gradient_bias
        
    def update(self, input=None, output=None):
        """Stochastic Gradient Descent update"""
        loss, gradient_weight, gradient_bias  = self.backpropagation(input, output)

        #
        self.weight = self.weight - self.learning_rate * gradient_weight
        self.bias = self.bias - self.learning_rate * gradient_bias
        return loss

learning_rate = 0.05
num_epochs = 10
batch_size = 30

model = NumpyLogLinear(
    input_size=corpus.nr_features,
    num_classes=2, 
    learning_rate=learning_rate
)

# Define number of epochs and batch size
num_epochs = 10
batch_size = 30

# Get batch iterators for train and test
train_batches = data.batches('train', batch_size=batch_size)
test_set = data.batches('test', batch_size=None)[0]

In [54]:
train_batches[0]["input"].shape

(30, 13989)

In [55]:
log_z_tilde = model.log_forward(train_batches[0]["input"])

In [69]:
log_z_tilde[np.arange(batch_size),train_batch["output"]]

array([-0.77672866, -0.10876523, -1.58735147, -0.56809016, -0.88780013,
       -0.10876523, -0.67399417, -0.3910237 , -1.58508493, -0.19735229,
       -2.38458124, -0.64805753, -2.06609988, -0.39387638, -0.8979168 ,
       -0.83946934, -0.71304713, -0.47465503, -0.34647679, -0.62892631,
       -0.17329283, -0.32626888, -1.53983547, -0.05027497, -0.7182678 ,
       -0.54655371, -0.71607379, -0.51192325, -1.72493869, -0.83404784])

In [67]:
train_batch["output"].shape

(30,)

In [66]:
log_z_tilde.shape

(30, 2)

In [56]:
train_batch = train_batches[0]
loss, gradient_weight, gradient_bias =model.backpropagation(train_batch["input"], train_batch["output"])

In [61]:
model.update(train_batch["input"], train_batch["output"])

In [71]:
# Epoch loop
for epoch in range(num_epochs):

    # Batch loop
    for batch in train_batches:
        loss = model.update(input=batch['input'], output=batch['output'])
        print("\t", loss)

    # Prediction for this epoch
    hat_y = model.predict(input=test_set['input'])

    # Evaluation
    accuracy = 100*np.mean(hat_y == test_set['output'])

    # Inform user
    print("Epoch %d: accuracy %2.2f %%" % (epoch+1, accuracy))

	 0.7806513201847287
	 0.7284230427639542
	 0.8552704525381954
	 0.7225755862536837
	 0.5400638410201223
	 0.8548233418615167
	 1.0405345966696413
	 0.7410438072190298
	 0.7102748125357318
	 0.7748242234314024
	 0.735857331044392
	 0.6333475311618181
	 0.7509664059161442
	 0.7582859925058638
	 0.9362422488666432
	 0.8610243069283366
	 0.762395140298427
	 0.7830900065724723
	 0.7547367372160622
	 0.7215020820913839
	 0.7603250960930954
	 0.6167282707557686
	 0.726962516625075
	 0.5613228458833941
	 0.7148168211466418
	 0.6596043396211263
	 0.6170789397810218
	 0.8010581764430528
	 0.6174860288418604
	 0.7734769798694778
	 0.8570110323789003
	 0.5872782110627295
	 0.6955264971497945
	 0.6056783955980445
	 0.8248137488238513
	 0.721216644240656
	 0.7292988235448686
	 0.6959337929643216
	 0.654165776615883
	 0.6893155218970144
	 0.7032918633567105
	 0.6283346276756429
	 0.5487631177455948
	 0.5387101755468479
	 0.49144421177629244
	 0.7473892592086876
	 0.5840857435478561
	 0.5654342632534

In [59]:
np.sum(gradient_bias,0)

array([-0.13679231,  0.13679231])

In [57]:
gradient_weight.shape

(2, 13989)

In [48]:
%debug

> /tmp/ipykernel_9152/4250036294.py(31)backpropagation()
     29         z_tilde = np.exp(log_z_tilde)
     30         I = index2onehot(output, self.weight.shape[0])
---> 31         gradient_weight = - ((I-z_tilde) @ input.T)/batch_size
     32         return loss, gradient_weight, gradient_bias
     33 



ipdb>  (I-z_tilde).shape


(30, 2)


ipdb>  input.shape


(30, 13989)


ipdb>  q
